# Sentinel-1 - SAR amplitude quicklooks

Sentinel-1 carries a radar, so a "band" is a **polarisation**: the transmit and
receive orientation of the pulse. Which ones exist depends on the acquisition
mode, and this trips people up:

| Mode token | Polarisations present |
|---|---|
| `1SDV` (IW dual, vertical) | **VV + VH** |
| `1SDH` (EW dual, horizontal) | **HH + HV** |
| `1SSV` / `1SSH` (single) | one band only |

Assuming VV/VH is a real bug - it fails outright on the EW/`1SDH` products that
cover the Norwegian Arctic. `pysent.s1.detect_sentinel_s1_polarizations()` reads
what is actually in the product instead of guessing.

The pipeline is: **warp** the GCP-referenced measurement band to a polar
stereographic grid, **stretch** amplitude to 8-bit grayscale plus an alpha mask,
**write** a tiled GeoTIFF with overviews.

In [ ]:
# --- papermill parameters -------------------------------------------------
# Leave everything as-is to run against the committed test fixtures (what CI
# does). Point any of these at real data for the full pipeline.
DATA_ROOT = ""      # mounted NBS archive, e.g. "/data/nbsArchive"
SAFE_PATH = ""      # explicit .SAFE / .zip product
IDENTIFIER = ""     # catalogue UUID, resolved via pysent.archive
ENDPOINT = None     # None -> NBS_SENTINEL_CSW_ENDPOINT / https://nbs.csw.met.no
OUTPUT_DIR = "_output"
PLATFORM = "S1"

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import rasterio

import pysent
import nbtools

print("pysent", pysent.__version__, "from", Path(pysent.__file__).parent)

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Find a product

In [ ]:
product = nbtools.resolve_input(
    PLATFORM,
    safe_path=SAFE_PATH or None,
    identifier=IDENTIFIER or None,
    data_root=DATA_ROOT or None,
    endpoint=ENDPOINT,
)
print(nbtools.describe(product))

## 3. Which polarisations are really in there?\nRead from the SAFE manifest's per-band `POLARIZATION` metadata - the same field the warp matches on - so the answer is what the warp will actually find.

In [ ]:
from pysent.s1 import S1_SUPPORTED_AMPLITUDE_VARIABLES, detect_sentinel_s1_polarizations

print("library supports:", S1_SUPPORTED_AMPLITUDE_VARIABLES)

if product.can_warp:
    present = detect_sentinel_s1_polarizations(str(product.path))
    print("present in this product:", present or "(could not read manifest)")
else:
    print("fixture mode: the extract came from", product.label)
    print("its mode token is", product.label.split("_")[3], "-> see the table above")

## 4. The amplitude stretch\nSAR amplitude is heavily skewed: mostly low backscatter with a long bright tail from urban areas and specular returns. A percentile clip keeps the bulk of the scene usable instead of letting a few bright pixels dominate.

In [ ]:
from pysent.s1 import S1_STRETCH_PERCENTILES, stretch_sentinel_s1_grayscale

with rasterio.open(product.path) as src:
    amplitude = src.read(1).astype(np.float32)
    nodata = src.nodata

gray, alpha, stats = stretch_sentinel_s1_grayscale(
    amplitude, nodata=nodata, percentiles=S1_STRETCH_PERCENTILES)

valid = amplitude > 0
print(f"amplitude   : {amplitude[valid].min():.0f} - {amplitude[valid].max():.0f} "
      f"(mean {amplitude[valid].mean():.0f})")
print(f"clip points : {stats['p_low']:.1f} -> 0,  {stats['p_high']:.1f} -> 255")
print(f"valid pixels: {valid.mean():.1%}  (alpha marks the rest transparent)")
print(f"output      : {gray.dtype}, {(gray[valid] == 0).mean():.1%} black, "
      f"{(gray[valid] == 255).mean():.1%} white")

## 5. Choosing the percentiles\nTwo failure modes pull in opposite directions. **Over-stretch** crushes detail into pure black/white; **under-stretch** wastes the 8-bit range and looks flat. Sweep the parameter and watch both at once.

In [ ]:
print(f"{'percentiles':>14} | {'clipped':>8} | {'range used':>10} | verdict")
print("-" * 56)
for percentiles in [(0.0, 100.0), (1.0, 99.0), (2.0, 98.0), (5.0, 95.0), (20.0, 80.0)]:
    g, _, _ = stretch_sentinel_s1_grayscale(amplitude, nodata=nodata, percentiles=percentiles)
    values = g[valid]
    clipped = ((values == 0) | (values == 255)).mean()
    p2, p98 = np.percentile(values, [2, 98])
    used = (p98 - p2) / 255
    verdict = "over-stretched" if clipped > 0.15 else ("dull" if used < 0.6 else "good")
    print(f"{str(percentiles):>14} | {clipped:7.1%} | {used:9.1%} | {verdict}")

Wide percentiles clip almost nothing but leave the image dull; narrow ones use the full range but destroy detail. The default `(2, 98)` sits where both numbers are acceptable.

### Why the warp dominates the cost

The measurement raster is not a neat north-up grid: it is referenced by **ground
control points**, and rectifying it is most of the processing time (~27 s of a
~31 s S1 quicklook in our benchmarks).

`_warp_sentinel_s1_safe_amplitude` takes `use_tps`:

- `use_tps=True` (default) - thin-plate spline through the GCPs. Most accurate,
  and the dominant cost.
- `use_tps=False` - GDAL's polynomial GCP transform. Substantially faster, with
  some geometric accuracy given up.

A note that cost us time: the SAFE band is **GCP**-referenced, not a geolocation
array, so `geoloc=True` does *not* apply here and fails with
`Unable to compute a GEOLOC_ARRAY based transformation`. That option belongs to
the NetCDF path only.

In [ ]:
if product.can_warp:
    from pysent.s1 import S1_TARGET_EPSG, S1_TARGET_RESOLUTION, process_sentinel_s1_safe

    variables = detect_sentinel_s1_polarizations(str(product.path))[:1]
    results = process_sentinel_s1_safe(
        input_dataset=str(product.path),
        output_dir=OUTPUT_DIR,
        output_names={v: f"quicklook_{v}.tif" for v in variables},
        processing_options={"target_epsg": S1_TARGET_EPSG,
                            "target_resolution": S1_TARGET_RESOLUTION,
                            "compression": "DEFLATE"},
    )
    for entry in results:
        print(entry)
else:
    print("Fixture mode - no SAFE to warp. Set DATA_ROOT/SAFE_PATH to run this cell.")

## 7. Tuning notes\n\n- **dB scaling is the biggest quality win still on the table.** Backscatter spans orders of magnitude; `20*log10(amplitude)` before the percentile clip usually gives markedly better contrast than the current linear stretch.\n- **`use_tps=False` is the biggest speed win**, since the warp is most of the cost.\n- **Speckle filtering** (Lee / refined-Lee) before the stretch is worth trying.\n\nMeasure any of these with [04_benchmarks.ipynb](04_benchmarks.ipynb).